Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
import glob
import os
from tqdm import tqdm

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein/vein001_1/01.jpg'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)

Preprocessing for Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm
import re

# ==== CONFIG ====
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'


TRAIN_IMAGES_PER_SESSION = 4  # Use 01.jpg to 04.jpg
IMAGE_SIZE = (100, 300)

train_data = []
train_labels = []

print("\n🚀 Preparing training data (Protocol 1, Strategy 2) using real folder names...\n")

# List all finger folders
folder_list = sorted([f for f in os.listdir(base_path_sess1) if f.startswith("vein")])

for folder_name in tqdm(folder_list, desc="Processing folders"):
    match = re.match(r"vein(\d{3})_(\d)", folder_name)
    if not match:
        print(f"⚠️ Skipping unrecognized folder: {folder_name}")
        continue

    subject_id = match.group(1)
    finger_id = match.group(2)

    for session_label, base_path in [("session1", base_path_sess1), ("session2", base_path_sess2)]:
        folder_path = os.path.join(base_path, folder_name)

        for i in range(1, TRAIN_IMAGES_PER_SESSION + 1):  # Use images 01–03
            img_filename = f"{i:02d}.jpg"
            img_path = os.path.join(folder_path, img_filename)

            print(f"🔍 {session_label} | Folder: {folder_name} | Image: {img_filename}")
            print(f"     → Path: {img_path}")

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"❌ Missing image: {img_path}")
                continue

            print("     ✅ Image loaded successfully")

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            train_data.append(img_norm.flatten())
            class_label = f"{session_label}_subject{subject_id}_fingervein{finger_id}"
            train_labels.append(class_label)
            print(f"     🏷️ Label assigned: {class_label}")

# Convert to arrays
train_data = np.array(train_data)
train_labels = np.array(train_labels)

print("\n✅ Data preparation complete!")
print("📐 Train data shape:", train_data.shape)
print("🏷️ Train labels shape:", train_labels.shape)
print("🔎 Example labels:", train_labels[:5])


Training:

In [ ]:

import numpy as np

# Step 1: Center the training data
mean_vector = np.mean(train_data, axis=0)
centered_data = train_data - mean_vector  # Shape: (n_samples, n_features)

# Step 2: Compute subject-to-subject Gram matrix
gram_matrix = centered_data @ centered_data.T  # Shape: (n_samples, n_samples)

# Step 3: Eigen decomposition of Gram matrix
eig_vals, eig_vecs = np.linalg.eigh(gram_matrix)  # ascending order

# Step 4: Sort eigenvalues/vectors in descending order
sorted_indices = np.argsort(-eig_vals)
eig_vals = eig_vals[sorted_indices]
eig_vecs = eig_vecs[:, sorted_indices]

# Step 5: Filter valid eigenvectors (non-zero eigenvalues)
valid_indices = eig_vals > 1e-10
eig_vals_valid = eig_vals[valid_indices]
eig_vecs_valid = eig_vecs[:, valid_indices]

# ✅ Step 6: Project all valid eigenvectors back to original feature space
eig_vecs_full = (centered_data.T @ eig_vecs_valid) / np.sqrt(eig_vals_valid)

# Step 7: Project training data into full PCA space
train_data_pca = centered_data @ eig_vecs_full  # Shape: (n_samples, k)

# Final output
print("✅ PCA-transformed training data shape:", train_data_pca.shape)
print("✅ Number of principal components used:", eig_vecs_full.shape[1])


Preprocessing for Testing:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm
import re

# Define session base paths
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

IMAGE_SIZE = (100, 300)
test_data = []
test_labels = []

print("\n🧪 Preparing test data (Protocol 1, Strategy 2 – using 04, 05, and 06 from both sessions)...\n")

# List actual folders from session 1 (assumes both sessions have the same folder structure)
folder_list = sorted([f for f in os.listdir(base_path_sess1) if f.startswith("vein")])

for folder_name in tqdm(folder_list, desc="Processing test folders"):
    match = re.match(r"vein(\d{3})_(\d)", folder_name)
    if not match:
        print(f"⚠️ Skipping unrecognized folder: {folder_name}")
        continue

    subject_id = match.group(1)
    finger_id = match.group(2)

    for session_label, base_path in [("session1", base_path_sess1), ("session2", base_path_sess2)]:
        folder_path = os.path.join(base_path, folder_name)

        for i in range(5, 7):  # ✅ Use 04.jpg, 05.jpg, and 06.jpg
            img_filename = f"{i:02d}.jpg"
            img_path = os.path.join(folder_path, img_filename)

            print(f"🔍 {session_label} | Folder: {folder_name} | Image: {img_filename}")
            print(f"     → Path: {img_path}")

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"❌ Missing image: {img_path}")
                continue

            print("     ✅ Image loaded successfully")
            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            test_data.append(img_norm.flatten())

            label = f"{session_label}_subject{subject_id}_fingervein{finger_id}"
            test_labels.append(label)
            print(f"     🏷️ Label assigned: {label}")

# Convert to numpy arrays
test_data = np.array(test_data)
test_labels = np.array(test_labels)

# Final stats
print("\n✅ Test data preparation complete!")
print("📐 Testing data shape:", test_data.shape)
print("🏷️ Testing labels shape:", test_labels.shape)
print("🔎 Example test labels:", test_labels[:5])


Test:

In [ ]:
correct_matches = 0
total_tests = len(test_data)  # Should be 1968 test samples

# Step 1: Center and project test data into PCA space
print("\n🔧 Step 1: Centering and projecting test data into PCA space...")
centered_test_data = test_data - mean_vector
proj_test_data = centered_test_data @ eig_vecs_full
print("✅ Projection complete.\n")

# Step 2: Compare each test sample with all training samples
print("🔍 Step 2: Matching test samples with nearest training samples...\n")

for i in range(total_tests):
    proj_test = proj_test_data[i]
    true_label = test_labels[i]

    # Compute Manhattan distances
    distances = np.sum(np.abs(train_data_pca - proj_test), axis=1)

    # Nearest match
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]

    # Output result
    if predicted_label == true_label:
        correct_matches += 1
        result = "✅ CORRECT"
        emoji = "🎯"
    else:
        result = "❌ WRONG"
        emoji = "⚠️"

    print(f"{emoji} Test sample {i+1}/{total_tests}")
    print(f"    🧾 Predicted: {predicted_label}")
    print(f"    🎯 Actual   : {true_label}")
    print(f"    ➡️  Result   : {result}\n")

# Step 3: Final accuracy
accuracy = (correct_matches / total_tests) * 100
print("📊 Final Results")
print(f"✅ Correct matches: {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy: {accuracy:.2f}%")
